In [14]:
#load dataset
with open("data/mypos-ver.3.0.txt", "r", encoding = "utf-8") as f:
    for i in range(5):
        print(f.readline())

ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc

အသစ်/n ဝယ်/v ထား/part တဲ့/part ဆွယ်တာ/n က/ppm အသီး/n ထ/v နေ/part ပါ/part ပေါ့/part ။/punc

မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc

ပေဟိုင်/n|ဥယျာဉ်/n ။/punc

နဝမ/adj အိပ်မက်/n ကောသလ/n|မင်း/n|အိပ်မက်/n ၉/num နက်ရှိုင်း/adj ကျယ်ဝန်း/adj သော/part ရေကန်/n ကြီး/adj တစ်/tn ခု/part တွင်/ppm သတ္တဝါ/n တို့/part ဆင်း/v ၍/conj ရေသောက်/v ကြ/part ၏/ppm ။/punc



Prepare dataset

In [15]:
def load_pos_data(filepath):
    sentences = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pairs = line.split()
            sentence = [tuple(p.rsplit('/', 1)) for p in pairs if '/' in p]
            if sentence:
                sentences.append(sentence)
    return sentences

sentences = load_pos_data("data/mypos-ver.3.0.txt")
print(sentences[0][:5])

[('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm'), ('၁၀၀', 'num'), ('ရာခိုင်နှုန်း', 'n')]


Feature Extraction

In [16]:
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word': word,
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word[:2]': word[:2],
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word': word1,
            '-1:word[-2:]': word1[-2:],
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word': word1,
            '+1:word[-2:]': word1[-2:],
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for token, tag in sent]

# quick test
print(sent2features(sentences[0])[0])
print(sent2labels(sentences[0]))

{'bias': 1.0, 'word': 'ဒီ', 'word[-3:]': 'ဒီ', 'word[-2:]': 'ဒီ', 'word[:2]': 'ဒီ', 'word.isdigit()': False, 'BOS': True, '+1:word': 'ဆေး', '+1:word[-2:]': 'ေး'}
['adj', 'n', 'ppm', 'num', 'n', 'adj', 'n', 'part', 'ppm', 'v', 'part', 'part', 'v', 'ppm', 'punc']


In [17]:
!pip install python-crfsuite

Defaulting to user installation because normal site-packages is not writeable


In [18]:
import random
random.seed(42)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train_sents = sentences[:split]
test_sents = sentences[split:]

print(len(train_sents), len(test_sents))

34556 8640


In [19]:
import pycrfsuite

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

trainer = pycrfsuite.Trainer(verbose=False)
for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

trainer.set_params({
    'c1': 1.0,
    'c2': 1e-3,
    'max_iterations': 100,
    'feature.possible_transitions': True
})

trainer.train('mypos_model.crfsuite')
print("Training done")

Training done


In [20]:
tagger = pycrfsuite.Tagger()
tagger.open('mypos_model.crfsuite')

y_pred = [tagger.tag(xseq) for xseq in X_test]

# quick look at one example
print("Words: ", [w for w, t in test_sents[0]])
print("True:  ", y_test[0])
print("Pred:  ", y_pred[0])

Words:  ['ရောက်ဘာပြား/n|ကြိတ်/v|စက်/n|လုပ်ငန်း', '၊', 'သံရည်ကျို/v|လုပ်ငန်း', '၊', 'သံအမာတင်/v|ခြင်း', 'လုပ်ငန်း', '၊', 'ဓာတ်မြေဩဇာ/n|ထုတ်လုပ်/v|မှု/part|လုပ်ငန်း', '၊', 'ရေ/n|လုပ်ငန်း/n|သုံး/v|ဘော့သီး/n|လုပ်ငန်း', '၊', 'မော်တော်ယာဉ်/n|ထုတ်လုပ်/v|ရေး/part|လုပ်ငန်း', 'စသည်', 'တို့', 'ကို', 'အဓိက', 'လုပ်ကိုင်', 'လျက်', 'ရှိ', 'ကြ', 'သည်', '။']
True:   ['n', 'punc', 'n', 'punc', 'part', 'n', 'punc', 'n', 'punc', 'n', 'punc', 'n', 'part', 'part', 'ppm', 'n', 'v', 'conj', 'v', 'part', 'ppm', 'punc']
Pred:   ['n', 'punc', 'n', 'punc', 'n', 'n', 'punc', 'n', 'punc', 'n', 'punc', 'n', 'part', 'part', 'ppm', 'n', 'v', 'conj', 'v', 'part', 'ppm', 'punc']


In [21]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [22]:
from sklearn.metrics import classification_report
y_test_flat = [tag for sent in y_test for tag in sent]
y_prep_flat = [tag for sent in y_pred for tag in sent]
print(classification_report(y_test_flat, y_prep_flat))

              precision    recall  f1-score   support

         abb       0.97      0.80      0.88        81
         adj       0.86      0.80      0.83      3178
         adv       0.90      0.82      0.86      2157
        conj       0.89      0.94      0.91      3545
          fw       0.98      0.89      0.93       723
         int       0.96      0.90      0.93       125
           n       0.94      0.96      0.95     21178
         num       0.99      0.98      0.98      1256
        part       0.96      0.96      0.96     26721
         ppm       0.98      0.98      0.98     17516
        pron       0.97      0.96      0.97      4140
        punc       1.00      1.00      1.00     10792
          sb       0.95      0.90      0.92        80
          tn       0.97      0.96      0.97      1125
           v       0.94      0.93      0.94     15455

    accuracy                           0.96    108072
   macro avg       0.95      0.92      0.93    108072
weighted avg       0.96   

In [23]:
new_sentence = ["သူ", "မနက်ဖြန်", "ကျာင်း", "သွား", "မယ်", "အိမ်", "ပြန်", "မယ်"]

features = sent2features([(w, None) for w in new_sentence])
predicted_tags = tagger.tag(features)

for word, tag in zip(new_sentence, predicted_tags):
    print(word, "->", tag)
    

သူ -> pron
မနက်ဖြန် -> n
ကျာင်း -> n
သွား -> v
မယ် -> ppm
အိမ် -> n
ပြန် -> v
မယ် -> ppm
